In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil dibuat!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/13 18:32:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat!


In [5]:
import os
import numpy as np
import pandas as pd

np.random.seed(42)
n = 600
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-07-01", "2026-07-31", freq="D")

data = {
    "order_id": [f"ORD-{1000+i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n, p=[0.25,0.25,0.20,0.15,0.15]),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000,50000,75000,100000,150000,250000,500000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n, p=[0.35,0.30,0.20,0.15]),
}

pd.DataFrame(data).to_csv("data_transaksi_ecommerce.csv", index=False)
print("File data_transaksi_ecommerce.csv BERHASIL dibuat!")

File data_transaksi_ecommerce.csv BERHASIL dibuat!


In [6]:
df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)

print("Tipe objek:", type(df))
df.printSchema()

Tipe objek: <class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)



In [8]:
df.show(5)

print("Jumlah baris:", df.count())

+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|ORD-1000|2026-07-07 00:00:00|Kesehatan & Kecan...|  Magelang|           9|      250000|         E-Wallet|
|ORD-1001|2026-07-20 00:00:00|   Makanan & Minuman|Yogyakarta|           9|       50000|     Kartu Kredit|
|ORD-1002|2026-07-29 00:00:00|             Fashion| Purworejo|           2|       50000|     Kartu Kredit|
|ORD-1003|2026-07-15 00:00:00|          Elektronik|      Solo|           3|       50000|         E-Wallet|
|ORD-1004|2026-07-11 00:00:00|          Elektronik|  Semarang|           5|      250000|    Transfer Bank|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
only showing top 5 rows

Jumlah baris

In [9]:
df.select("order_id", "kota", "kategori").show(5)

+--------+----------+--------------------+
|order_id|      kota|            kategori|
+--------+----------+--------------------+
|ORD-1000|  Magelang|Kesehatan & Kecan...|
|ORD-1001|Yogyakarta|   Makanan & Minuman|
|ORD-1002| Purworejo|             Fashion|
|ORD-1003|      Solo|          Elektronik|
|ORD-1004|  Semarang|          Elektronik|
+--------+----------+--------------------+
only showing top 5 rows



In [11]:
from pyspark.sql.functions import col

df.filter((col("kategori") == "Elektronik") & (col("unit_terjual") > 5)).show(5)

+--------+-------------------+----------+----------+------------+------------+-----------------+
|order_id|            tanggal|  kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+----------+----------+------------+------------+-----------------+
|ORD-1019|2026-07-03 00:00:00|Elektronik|  Magelang|           9|       75000|         E-Wallet|
|ORD-1020|2026-07-22 00:00:00|Elektronik|  Semarang|           6|      150000|     Kartu Kredit|
|ORD-1028|2026-07-28 00:00:00|Elektronik|  Semarang|           8|      250000|    Transfer Bank|
|ORD-1040|2026-07-31 00:00:00|Elektronik| Purworejo|           6|      100000|         E-Wallet|
|ORD-1054|2026-07-25 00:00:00|Elektronik|Yogyakarta|           9|      500000|    Transfer Bank|
+--------+-------------------+----------+----------+------------+------------+-----------------+
only showing top 5 rows



In [12]:
from pyspark.sql.functions import sum as spark_sum, count, avg

# Menambahkan kolom baru: total_pendapatan = unit_terjual x harga_satuan
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# Meringkas: total pendapatan & jumlah transaksi per kota, diurutkan dari tertinggi
ringkasan_kota = df.groupBy("kota").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan"),
    count("order_id").alias("jumlah_transaksi"),
    avg("unit_terjual").alias("rata_rata_unit")
).orderBy(col("total_pendapatan").desc())

ringkasan_kota.show()

+----------+----------------+----------------+-----------------+
|      kota|total_pendapatan|jumlah_transaksi|   rata_rata_unit|
+----------+----------------+----------------+-----------------+
|  Magelang|       109925000|             119| 5.07563025210084|
| Purworejo|       104675000|             122|4.737704918032787|
|Yogyakarta|       104625000|             129|5.248062015503876|
|  Semarang|        87200000|             118|4.771186440677966|
|      Solo|        83950000|             112|5.053571428571429|
+----------+----------------+----------------+-----------------+



In [13]:
!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan4
!hdfs dfs -put -f data_transaksi_ecommerce.csv /user/mahasiswa/pertemuan4/

mkdir: Call From azka-VivoBook-14-ASUS-Laptop-X407UAR/127.0.1.1 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused
put: Call From azka-VivoBook-14-ASUS-Laptop-X407UAR/127.0.1.1 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused


In [6]:
df_dari_hdfs = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan4/data_transaksi_ecommerce.csv",
    header=True, inferSchema=True
)
print("Jumlah baris dari HDFS:", df_dari_hdfs.count())
df_dari_hdfs.show(5)

In [4]:
!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan4
!hdfs dfs -put -f data_transaksi_ecommerce.csv /user/mahasiswa/pertemuan4/

In [7]:
!hdfs dfs -ls /user/mahasiswa/pertemuan4/

Found 1 items
-rw-r--r--   1 azka supergroup      46518 2026-09-13 18:35 /user/mahasiswa/pertemuan4/data_transaksi_ecommerce.csv


In [8]:
import os
import numpy as np
import pandas as pd

# 1. Buat file CSV di lokal jika belum ada
n = 600
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-07-01", "2026-07-31", freq="D")

data = {
    "order_id": [f"ORD-{1000+i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000,50000,75000,100000,150000,250000,500000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
}
pd.DataFrame(data).to_csv("data_transaksi_ecommerce.csv", index=False)

# 2. Upload ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan4
!hdfs dfs -put -f data_transaksi_ecommerce.csv /user/mahasiswa/pertemuan4/
!hdfs dfs -ls /user/mahasiswa/pertemuan4/

Found 1 items
-rw-r--r--   1 azka supergroup      46827 2026-09-13 18:37 /user/mahasiswa/pertemuan4/data_transaksi_ecommerce.csv


In [9]:
df_dari_hdfs = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan4/data_transaksi_ecommerce.csv",
    header=True, inferSchema=True
)
print("Jumlah baris dari HDFS:", df_dari_hdfs.count())
df_dari_hdfs.show(5)